In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

pd.set_option("display.max_columns",50)
pd.set_option("display.width",200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

#Shared colour pallete so every chart reads consistently
PALETTE =   px.colors.qualitative.Bold
PRIMARY = "#1f77b4"
ACCENT = "#FF7F0E"

In [4]:
cleaned_customer_transaction = pd.read_csv("../customer_data/artifacts/cleaned_transaction_data.csv")

In [5]:
cleaned_customer_transaction.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,Traditional,2007-04-30,Male,London,"4,021.65",0,Completed


In [6]:
cleaned_customer_transaction.info()

<class 'pandas.DataFrame'>
RangeIndex: 580962 entries, 0 to 580961
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   TransactionID        580962 non-null  str    
 1   CustomerID           580962 non-null  str    
 2   TransactionDate      580962 non-null  str    
 3   TransactionTime      580962 non-null  str    
 4   TransactionType      580962 non-null  str    
 5   TransactionAmount    580962 non-null  float64
 6   TransactionCurrency  580962 non-null  str    
 7   AssetType            580962 non-null  str    
 8   CustomerDOB          580962 non-null  str    
 9   CustGender           580962 non-null  str    
 10  CustLocation         580962 non-null  str    
 11  CustAccountBalance   580962 non-null  float64
 12  WalletActivity       580962 non-null  int64  
 13  TransactionStatus    580962 non-null  str    
dtypes: float64(2), int64(1), str(11)
memory usage: 114.6 MB


In [7]:
cleaned_customer_transaction.isna().sum()

TransactionID          0
CustomerID             0
TransactionDate        0
TransactionTime        0
TransactionType        0
TransactionAmount      0
TransactionCurrency    0
AssetType              0
CustomerDOB            0
CustGender             0
CustLocation           0
CustAccountBalance     0
WalletActivity         0
TransactionStatus      0
dtype: int64

In [10]:
#Parse  the data once, reuse everywhere 
cleaned_customer_transaction["TransactionDate"] = pd.to_datetime(cleaned_customer_transaction["TransactionDate"])
cleaned_customer_transaction["CustomerDOB"]= pd.to_datetime(cleaned_customer_transaction["CustomerDOB"])
cleaned_customer_transaction["TransactionTime"] = pd.to_datetime(cleaned_customer_transaction["TransactionTime"], format = "%H:%M:%S").dt.time

print(f"Rows: {len(cleaned_customer_transaction):,}")
print(f"Columns: {cleaned_customer_transaction.shape[1]}")
print(f"Customers: {cleaned_customer_transaction["CustomerID"].nunique():,}")
print(f"Date range: {cleaned_customer_transaction["TransactionDate"].min().date()} - {cleaned_customer_transaction["TransactionDate"].max().date()}")
print(f"Total nulls: {cleaned_customer_transaction.isna().sum().sum()}")

Rows: 580,962
Columns: 14
Customers: 4,997
Date range: 2024-12-13 - 2026-09-12
Total nulls: 0


In [11]:
cleaned_customer_transaction["TransactionStatus"].value_counts()

TransactionStatus
Completed       544678
Pending          23325
Failed            5517
Under Review      4170
Rejected          3272
Name: count, dtype: int64

In [ ]:
# GBP equivalent amount (needed formonetary charts;cryptorows are in asset units)
FX_TO_GBP = {"GBP": 1.0, "EUR": 0.85, "USD": 0.78, "USDT": 0.78,
             "BTC": 42_000.0, "ETH": 2_300.0}

cleaned_customer_transaction["AmountGBP"]= cleaned_customer_transaction["TransactionAmount"] * cleaned_customer_transaction["TransactionCurrency"].map(FX_TO_GBP)

cleaned_customer_transaction["RevenueGBP"] = np.where(
    cleaned_customer_transaction["TransactionStatus"] == "Completed",
    cleaned_customer_transaction["AmountGBP"],
    0.0,
)

SNAPSHOT = cleaned_customer_transaction["TransactionDate"].max()

cleaned_customer_transaction["Year"] = cleaned_customer_transaction["TransactionDate"].dt.year
cleaned_customer_transaction["Month"] = cleaned_customer_transaction["TransactionDate"].dt.to_period("M").astype(str)
cleaned_customer_transaction["DayOfWeek"] = cleaned_customer_transaction["TransactionDate"].dt.day_name()
cleaned_customer_transaction["Hour"] = cleaned_customer_transaction["TransactionTime"].apply(;ambda t: t.hour)

cleaned_customer_transaction["Age"] = ((SNAPSHOT - cleaned_customer_transaction["CustomerDOB"]).dt.days / 365.25).round(1)

In [14]:
cleaned_customer_transaction.head(5)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus,AmountGBP,RevenueGBP,Year
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed,79.36,79.36,2024
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed,54.95,54.95,2024
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed,410.02,410.02,2024
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed,165.77,165.77,2024
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed,674.77,674.77,2024
